# Time-MoE

In [ ]:
!pip install matplotlib

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


In [ ]:
!pip install pyyaml
!pip install numpy
!pip install pandas
!pip install scikit-learn

In [ ]:
!pip install transformers==4.40.1

In [ ]:
#!pip install datasets==2.18.0

In [ ]:
!pip install accelerate==0.28.0

In [ ]:
!pip install accelerate

## Imports

In [ ]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
from transformers import AutoModelForCausalLM

import joblib

from sklearn.metrics import r2_score

In [ ]:
def timemoe_forecast(
    df,
    target_column,
    context_length,
    prediction_length,
    test_size,
    model_size='50M',
    device = 'cpu'
):
    data = torch.tensor(df[target_column].values, dtype=torch.float32).to(device)
    
    model = AutoModelForCausalLM.from_pretrained(
        f'Maple728/TimeMoE-{model_size}',
        device_map=device,
        trust_remote_code=True
    )
    
    all_predictions = []
    
    with torch.no_grad():
        for i in range(0, test_size - prediction_length + 1, prediction_length):
            # Get sequence for current window
            start_idx = len(data) - test_size + i - context_length
            sequence = data[start_idx:start_idx + context_length]
            sequence = sequence.unsqueeze(0)  # Add batch dimension
            
            # Normalize sequence
            mean = sequence.mean(dim=-1, keepdim=True)
            std = sequence.std(dim=-1, keepdim=True)
            normalized_sequence = (sequence - mean) / std
            
            # Generate forecast
            output = model.generate(
                normalized_sequence, 
                max_new_tokens=prediction_length
            )
            
            # Denormalize predictions
            normed_preds = output[:, -prediction_length:]
            predictions = normed_preds * std + mean
            all_predictions.append(predictions.squeeze(0).cpu())
    
    return torch.cat(all_predictions).numpy()

In [ ]:
# Read the dataset
aquifer_by_stations = joblib.load('aquifer_by_stations.joblib')
aquifers_list = [85065]

In [ ]:
horizon = 5 # prediction horizon
day_len = 200 # number of days to forecast

# List for r2 results for different prediction horizons
r2_scores = [[] for _ in range(horizon)]

for aquifer in aquifers_list:
    # List for storing the predictions
    predictions = [[] for _ in range(5)]

    # Iterate from day_len days before the end, to the last day
    for i in range(day_len + (horizon-1), 0, -1):
        y = aquifer_by_stations[aquifer]

        forecast = timemoe_forecast(
            df=y,
            target_column='altitude_diff',
            context_length=6*horizon,
            prediction_length=horizon,
            test_size=i,
            device='cuda'
        )

        # Store the results for every prediction horizon separately
        for i in range(horizon):
            #print(forecast.head())
            predictions[i].append(forecast[i])
    
    # Clean up the results
    predictions[0] = predictions[0][-200:]
    predictions[1] = predictions[1][3:-1]
    predictions[2] = predictions[2][2:-2]
    predictions[3] = predictions[3][1:-3]
    predictions[4] = predictions[4][0:-4]

    # Calculate the r2 scores and store them in a list
    for i in range(horizon):
        r2_scores[i].append(r2_score(aquifer_by_stations[aquifer]['altitude_diff'][-day_len:], predictions[i]))

## Try number 2

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    'Maple728/TimeMoE-200M',
    device_map="cpu",  # use "cpu" for CPU inference, and "cuda" for GPU inference.
    trust_remote_code=True,
)

In [ ]:
# Read the dataset
aquifer_by_stations = joblib.load('aquifer_by_stations.joblib')
aquifers_list = [85065, 85064]

In [ ]:
horizon = 5 # prediction horizon
day_len = 365 # number of days to forecast
context_length = 730

# List for r2 results for different prediction horizons
r2_scores = [[] for _ in range(horizon)]

for aquifer in aquifers_list:
    # List for storing the predictions
    predictions = [[] for _ in range(5)]

    with torch.no_grad():
        # Iterate from day_len days before the end, to the last day
        for j in range(day_len + (horizon-1), 0, -1):
            y = aquifer_by_stations[aquifer][-(j + context_length):-j]

            # Normalize the data
            mean, std = y['altitude_diff'].mean(), y['altitude_diff'].std()
            y['altitude_diff'] = (y['altitude_diff'] - mean) / std
            
            # Convert to tensor and add batch dimension, ensuring float32 dtype
            input_data = torch.tensor(y['altitude_diff'].values, dtype=torch.float32).unsqueeze(0)
            
            forecast = model.generate(
                inputs=input_data,
                max_new_tokens=horizon
            )
            
            # Convert back to numpy array
            forecast = forecast[0][-horizon:].cpu().numpy()
            forecast = forecast * std + mean

            # Store the results for every prediction horizon separately
            for i in range(horizon):
                predictions[i].append(forecast[i])
    
    # Clean up the results
    predictions[0] = predictions[0][-day_len:]
    predictions[1] = predictions[1][3:-1]
    predictions[2] = predictions[2][2:-2]
    predictions[3] = predictions[3][1:-3]
    predictions[4] = predictions[4][0:-4]

    # Calculate the r2 scores and store them in a list
    for i in range(horizon):
        r2_scores[i].append(r2_score(aquifer_by_stations[aquifer]['altitude_diff'][-day_len:], predictions[i]))

In [ ]:
# Calculate the average r2 score
r2_average =  []
std_dev = []

for i in range(horizon):
    r2_average.append(np.mean(r2_scores[i]))
    std_dev.append(np.std(r2_scores[i]))

In [ ]:
r2_average

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(aquifer_by_stations[aquifer]['date'][-day_len:], aquifer_by_stations[aquifer]['altitude_diff'][-day_len:], color="royalblue", label="true data")
plt.plot(aquifer_by_stations[aquifer]['date'][-day_len:], predictions[0], color="tomato", label="forecast0")
#plt.plot(aquifer_by_stations[aquifer]['date'][-day_len:], predictions[1], color="orange", label="forecast1")
#plt.plot(aquifer_by_stations[aquifer]['date'][-day_len:], predictions[2], color="green", label="forecast2")
#plt.plot(aquifer_by_stations[aquifer]['date'][-day_len:], predictions[3], color="purple", label="forecast3")
#plt.plot(aquifer_by_stations[aquifer]['date'][-day_len:], predictions[4], color="brown", label="forecast4")
plt.legend()
plt.grid()
plt.show()

## Multivariate

### Time-MoE first, Linear regression second

In [ ]:
# Get the data
aquifer_by_stations = joblib.load('../data/interim/ground-water-and-weather-with-forecasts-and-additional-features.joblib')

# Transform date column to year, month and day columns
for key in aquifer_by_stations.keys():
    aquifer_by_stations[key]['year'] = aquifer_by_stations[key]['date'].dt.year
    aquifer_by_stations[key]['month'] = aquifer_by_stations[key]['date'].dt.month
    aquifer_by_stations[key]['day'] = aquifer_by_stations[key]['date'].dt.day

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    'Maple728/TimeMoE-200M',
    device_map="cpu",  # use "cpu" for CPU inference, and "cuda" for GPU inference.
    trust_remote_code=True,
)

aquifers_list = [85065, 85064]

In [ ]:
horizon = 5 # prediction horizon
day_len = 7 * 365 # number of days to forecast
max_context_length = 730

# List for r2 results for different prediction horizons
r2_scores = [[] for _ in range(horizon)]

for aquifer in aquifers_list:
    # List for storing the predictions
    predictions = [[] for _ in range(5)]

    with torch.no_grad():
        # Iterate from day_len days before the end, to the last day
        for j in range(day_len + (horizon-1), -1, -1):
            context_length = min(aquifer_by_stations[aquifer].shape[0] - j, max_context_length)
            if j != 0:
                y = aquifer_by_stations[aquifer][-(j + context_length):-j]
            else:
                y = aquifer_by_stations[aquifer][-(j + context_length):]

            # Normalize the data
            mean, std = y['altitude_diff'].mean(), y['altitude_diff'].std()
            y['altitude_diff'] = (y['altitude_diff'] - mean) / std
            
            # Convert to tensor and add batch dimension, ensuring float32 dtype
            input_data = torch.tensor(y['altitude_diff'].values, dtype=torch.float32).unsqueeze(0)
            
            forecast = model.generate(
                inputs=input_data,
                max_new_tokens=horizon
            )
            
            # Convert back to numpy array
            forecast = forecast[0][-horizon:].cpu().numpy()
            forecast = forecast * std + mean

            # Store the results for every prediction horizon separately
            for i in range(horizon):
                predictions[i].append(forecast[i])
    
    # Clean up the results
    predictions[0] = predictions[0][-day_len:]
    predictions[1] = predictions[1][-day_len:]
    predictions[2] = predictions[2][-day_len:]
    predictions[3] = predictions[3][-day_len:]
    predictions[4] = predictions[4][-day_len:]

    # Calculate the r2 scores and store them in a list
    for i in range(horizon):
        r2_scores[i].append(r2_score(aquifer_by_stations[aquifer]['altitude_diff'][-day_len:], predictions[i]))

    # Shorten the data length to the prediction length
    aquifer_by_stations[aquifer] = aquifer_by_stations[aquifer][-day_len:]

    # Add the predictions to the dataframe
    for i in range(horizon):
        aquifer_by_stations[aquifer][f'forecast_altitude_diff_{i}'] = predictions[i]

In [ ]:
stations_list = [85065, 85064]
joblib.dump({key: aquifer_by_stations[key] for key in stations_list}, '../data/interim/ground-water-and-weather-with-weather-and-timemoe-forecasts-and-additional-features.joblib')

In [ ]:
# Linear regression for the multivariate model

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler


In [ ]:
horizon_max = 5
day_len = 365

model = LinearRegression()
aquifer_by_stations = joblib.load('../data/interim/ground-water-and-weather-with-weather-and-timemoe-forecasts-and-additional-features.joblib')
#aquifer_by_stations = joblib.load('../data/interim/ground-water-and-weather-with-forecasts-and-additional-features.joblib')

# List for r2 results for different prediction horizons
r2_scores = [[] for _ in range(horizon_max)]

# Test features
test_features = ['forecast_altitude_diff_0', 'forecast_altitude_diff_1', 'forecast_altitude_diff_2', 'forecast_altitude_diff_3', 'forecast_altitude_diff_4']
best_features = joblib.load('../data/interim/linear-regression-best-features.joblib')

# Dictionary for storing the predictions for all of the stations
predictions_by_stations = {key: [] for key in aquifers_list}

for aquifer in aquifers_list:
    predictions = []

    for horizon in range (1, horizon_max+1, 1):
        # Define the train and test set
        #X_train = aquifer_by_stations[aquifer][best_features[f'horizon_{horizon}']][:-(day_len + horizon)]
        #X_train = aquifer_by_stations[aquifer][:-(day_len + horizon)].drop(columns=['date', 'station_id', 'id', 'location_id'])
        X_train = aquifer_by_stations[aquifer][test_features + best_features[f'horizon_{horizon}']][:-(day_len + horizon)]
        y_train = aquifer_by_stations[aquifer]['altitude_diff'][horizon:-day_len]

        #X_test = aquifer_by_stations[aquifer][best_features[f'horizon_{horizon}']][-(day_len + horizon):-horizon]
        #X_test = aquifer_by_stations[aquifer][-(day_len + horizon):-horizon].drop(columns=['date', 'station_id', 'id', 'location_id'])
        X_test = aquifer_by_stations[aquifer][test_features + best_features[f'horizon_{horizon}']][-(day_len + horizon):-horizon]
        y_test = aquifer_by_stations[aquifer]['altitude_diff'][-day_len:]
        
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()

        # Scale the features
        X_train = pd.DataFrame(scaler_X.fit_transform(X_train), columns=X_train.columns)
        X_test = pd.DataFrame(scaler_X.transform(X_test), columns=X_test.columns)
        
        y_train = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()

        # Train the model
        model.fit(X_train, y_train)

        # Make predictions
        forecast = model.predict(X_test).tolist()
        # Flatten
        forecast = np.ravel(forecast)

        # Unscale the predictions
        forecast = scaler_y.inverse_transform(forecast.reshape(-1, 1)).ravel()

        # Store to the predictions
        predictions.append(forecast)
        
        # Calculate and save the r2 score
        r2_scores[horizon-1].append(r2_score(y_test, forecast))

    # Store the predictions to the dictionary
    predictions_by_stations[aquifer] = predictions

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(aquifer_by_stations[aquifer]['date'][-200:], aquifer_by_stations[aquifer]['altitude_diff'][-200:], color="royalblue", label="true data")
plt.plot(aquifer_by_stations[aquifer]['date'][-200:], predictions[4][-200:], color="tomato", label="forecast")
#plt.plot(aquifer_by_stations[aquifer]['date'][-200:], predictions[2][-200:], color="green", label="forecast")
#plt.plot(aquifer_by_stations[aquifer]['date'][-200:], predictions[4][-200:], color="grey", label="forecast")
#plt.plot(aquifer_by_stations[aquifer]['date'][-200:], aquifer_by_stations[aquifer]['precipitation_probability_max'][-200:].apply(lambda x: x/20), color="brown", label="forecast")
#plt.plot(aquifer_by_stations[aquifer]['date'][-200:], aquifer_by_stations[aquifer]['precipitation'][-200:].apply(lambda x: x/130), color="olive", label="forecast")
plt.plot(aquifer_by_stations[aquifer]['date'][-200:], predictions_by_stations[85064][0][-200:], color="grey", label="forecast")
#plt.savefig('../data/interim/plot.svg', format='svg')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Calculate the average r2 score
r2_average =  []
std_dev = []

for i in range(5):
    r2_average.append(np.mean(r2_scores[i]))
    std_dev.append(np.std(r2_scores[i]))

In [ ]:
r2_average

### Linear regression first, Time-MoE second


Linear regression

In [ ]:
# Get the data
aquifer_by_stations = joblib.load('../data/interim/ground-water-and-weather-with-forecasts-and-additional-features.joblib')

# Transform date column to year, month and day columns
for key in aquifer_by_stations.keys():
    aquifer_by_stations[key]['year'] = aquifer_by_stations[key]['date'].dt.year
    aquifer_by_stations[key]['month'] = aquifer_by_stations[key]['date'].dt.month
    aquifer_by_stations[key]['day'] = aquifer_by_stations[key]['date'].dt.day

In [ ]:
horizon_max = 5
day_len = 3 * 365 + 50

model = LinearRegression()
#aquifer_by_stations = joblib.load('../data/interim/ground-water-and-weather-with-weather-and-timemoe-forecasts-and-additional-features.joblib')
aquifer_by_stations = joblib.load('../data/interim/ground-water-and-weather-with-forecasts-and-additional-features.joblib')

# List for r2 results for different prediction horizons
r2_scores = [[] for _ in range(horizon_max)]


# Dictionary for storing the predictions for all of the stations
predictions_by_stations = {key: [] for key in aquifers_list}

for aquifer in aquifers_list:
    predictions = []

    for horizon in range (1, horizon_max+1, 1):
        # Define the train and test set
        #X_train = aquifer_by_stations[aquifer][best_features[f'horizon_{horizon}']][:-(day_len + horizon)]
        X_train = aquifer_by_stations[aquifer][:-(day_len + horizon)].drop(columns=['date', 'station_id', 'id', 'location_id'])
        #X_train = aquifer_by_stations[aquifer][test_features + best_features[f'horizon_{horizon}']][:-(day_len + horizon)]
        y_train = aquifer_by_stations[aquifer]['altitude_diff'][horizon:-day_len]

        #X_test = aquifer_by_stations[aquifer][best_features[f'horizon_{horizon}']][-(day_len + horizon):-horizon]
        X_test = aquifer_by_stations[aquifer][-(day_len + horizon):-horizon].drop(columns=['date', 'station_id', 'id', 'location_id'])
        #X_test = aquifer_by_stations[aquifer][test_features + best_features[f'horizon_{horizon}']][-(day_len + horizon):-horizon]
        y_test = aquifer_by_stations[aquifer]['altitude_diff'][-day_len:]
        
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()

        # Scale the features
        X_train = pd.DataFrame(scaler_X.fit_transform(X_train), columns=X_train.columns)
        X_test = pd.DataFrame(scaler_X.transform(X_test), columns=X_test.columns)
        
        y_train = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()

        # Train the model
        model.fit(X_train, y_train)

        # Make predictions
        forecast = model.predict(X_test).tolist()
        # Flatten
        forecast = np.ravel(forecast)

        # Unscale the predictions
        forecast = scaler_y.inverse_transform(forecast.reshape(-1, 1)).ravel()

        # Store to the predictions
        predictions.append(forecast)
        
        # Calculate and save the r2 score
        r2_scores[horizon-1].append(r2_score(y_test, forecast))

    # Store the predictions to the dictionary
    predictions_by_stations[aquifer] = predictions

In [ ]:
# Shorten the data length to the prediction length
for aquifer in aquifers_list:
    aquifer_by_stations[aquifer] = aquifer_by_stations[aquifer][-day_len:]

# A new dataframe with residuals
residuals = {key: [] for key in aquifers_list}

for aquifer in aquifers_list:
    dataframe = pd.DataFrame()
    for i in range(horizon_max):
        dataframe[f'residual_{i}'] = aquifer_by_stations[aquifer]['altitude_diff'] - predictions_by_stations[aquifer][i]
    residuals[aquifer] = dataframe

Time-MoE

In [ ]:
horizon_max = 5 # prediction horizon
day_len = 365 # number of days to forecast
context_length = 730
aquifers_list = [85065, 85064]

# Load the model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForCausalLM.from_pretrained(
    'Maple728/TimeMoE-200M',
    device_map=device,
    trust_remote_code=True,
).to(device)

# List for r2 results for different prediction horizons
r2_scores = [[] for _ in range(horizon_max)]

for aquifer in aquifers_list:
    # List for storing the predictions
    predictions = [[] for _ in range(horizon_max)]

    for horizon in range(1, horizon_max+1, 1):
        with torch.no_grad():
            # Iterate from day_len days before the end, to the last day
            for j in range(day_len + (horizon-1), (horizon-1), -1):
                y = residuals[aquifer][f'residual_{horizon-1}'][-(j + context_length):-j]

                # Normalize the data
                mean, std = y.mean(), y.std()
                y = (y - mean) / std
                
                # Convert to tensor, add batch dimension, ensure float32 dtype, and move to device
                input_data = torch.tensor(y.values, dtype=torch.float32).unsqueeze(0).to(device)
                
                forecast = model.generate(
                    inputs=input_data,
                    max_new_tokens=horizon
                )
                
                # Convert back to numpy array
                forecast = forecast[0][-horizon:].cpu().numpy()
                forecast = forecast * std + mean

                # Store the results for every prediction horizon separately
                predictions[horizon-1].append(forecast[horizon-1])
    
    # Transform the residuals to the predictions
    for i in range(horizon_max):
        predictions[i] = predictions_by_stations[aquifer][i][-day_len:] + np.array(predictions[i])

    # Calculate the r2 scores and store them in a list
    for i in range(horizon_max):
        r2_scores[i].append(r2_score(aquifer_by_stations[aquifer]['altitude_diff'][-day_len:], predictions[i]))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(aquifer_by_stations[aquifer]['date'][-200:], aquifer_by_stations[aquifer]['altitude_diff'][-200:], color="royalblue", label="true data")
plt.plot(aquifer_by_stations[aquifer]['date'][-200:], predictions[4][-200:], color="tomato", label="forecast")
#plt.plot(aquifer_by_stations[aquifer]['date'][-200:], predictions[2][-200:], color="green", label="forecast")
#plt.plot(aquifer_by_stations[aquifer]['date'][-200:], predictions[4][-200:], color="grey", label="forecast")
#plt.plot(aquifer_by_stations[aquifer]['date'][-200:], aquifer_by_stations[aquifer]['precipitation_probability_max'][-200:].apply(lambda x: x/20), color="brown", label="forecast")
#plt.plot(aquifer_by_stations[aquifer]['date'][-200:], aquifer_by_stations[aquifer]['precipitation'][-200:].apply(lambda x: x/130), color="olive", label="forecast")
plt.plot(aquifer_by_stations[aquifer]['date'][-200:], predictions_by_stations[85064][0][-200:], color="grey", label="forecast")
#plt.savefig('../data/interim/plot.svg', format='svg')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Calculate the average r2 score
r2_average =  []
std_dev = []

for i in range(5):
    r2_average.append(np.mean(r2_scores[i]))
    std_dev.append(np.std(r2_scores[i]))

In [ ]:
r2_average

### Linear regression + Time-MoE then Linear regression

In [ ]:
# Get the data
aquifer_by_stations = joblib.load('../data/interim/ground-water-and-weather-with-forecasts-and-additional-features.joblib')

# Transform date column to year, month and day columns
for key in aquifer_by_stations.keys():
    aquifer_by_stations[key]['year'] = aquifer_by_stations[key]['date'].dt.year
    aquifer_by_stations[key]['month'] = aquifer_by_stations[key]['date'].dt.month
    aquifer_by_stations[key]['day'] = aquifer_by_stations[key]['date'].dt.day

In [ ]:
horizon_max = 5
day_len = 2 * 365
model = LinearRegression()

# List for r2 results for different prediction horizons
r2_scores = [[] for _ in range(horizon_max)]


# Dictionary for storing the predictions for all of the stations
predictions_by_stations = {key: [] for key in aquifers_list}

for aquifer in aquifers_list:
    predictions = []

    for horizon in range (1, horizon_max+1, 1):
        # Define the train and test set
        #X_train = aquifer_by_stations[aquifer][best_features[f'horizon_{horizon}']][:-(day_len + horizon)]
        X_train = aquifer_by_stations[aquifer][:-(day_len + horizon)].drop(columns=['date', 'station_id', 'id', 'location_id'])
        #X_train = aquifer_by_stations[aquifer][test_features + best_features[f'horizon_{horizon}']][:-(day_len + horizon)]
        y_train = aquifer_by_stations[aquifer]['altitude_diff'][horizon:-day_len]

        #X_test = aquifer_by_stations[aquifer][best_features[f'horizon_{horizon}']][-(day_len + horizon):-horizon]
        X_test = aquifer_by_stations[aquifer][-(day_len + horizon):-horizon].drop(columns=['date', 'station_id', 'id', 'location_id'])
        #X_test = aquifer_by_stations[aquifer][test_features + best_features[f'horizon_{horizon}']][-(day_len + horizon):-horizon]
        y_test = aquifer_by_stations[aquifer]['altitude_diff'][-day_len:]
        
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()

        # Scale the features
        X_train = pd.DataFrame(scaler_X.fit_transform(X_train), columns=X_train.columns)
        X_test = pd.DataFrame(scaler_X.transform(X_test), columns=X_test.columns)
        
        y_train = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()

        # Train the model
        model.fit(X_train, y_train)

        # Make predictions
        forecast = model.predict(X_test).tolist()
        # Flatten
        forecast = np.ravel(forecast)

        # Unscale the predictions
        forecast = scaler_y.inverse_transform(forecast.reshape(-1, 1)).ravel()

        # Store to the predictions
        predictions.append(forecast)
        
        # Calculate and save the r2 score
        r2_scores[horizon-1].append(r2_score(y_test, forecast))

    # Store the predictions to the dictionary
    predictions_by_stations[aquifer] = predictions

    linear_predictions = predictions_by_stations.copy()

In [ ]:
horizon_max = 5
day_len = 365
aquifers_list = [85065, 85064]

model = LinearRegression()
aquifer_by_stations = joblib.load('../data/interim/ground-water-and-weather-with-weather-and-timemoe-forecasts-and-additional-features.joblib')

# List for r2 results for different prediction horizons
r2_scores = [[] for _ in range(horizon_max)]

# Test features
test_features = ['forecast_altitude_diff_0', 'forecast_altitude_diff_1', 'forecast_altitude_diff_2', 'forecast_altitude_diff_3', 'forecast_altitude_diff_4']

# Dictionary for storing the predictions for all of the stations
predictions_by_stations = {key: [] for key in aquifers_list}

for aquifer in aquifers_list:
    predictions = []

    for horizon in range (1, horizon_max+1, 1):
        # Define the train and test set
        temp = pd.DataFrame({'a': aquifer_by_stations[aquifer][test_features[horizon-1]][-(2 * day_len + horizon):-(horizon)].values,
                             'b': linear_predictions[aquifer][horizon-1],
                             'target': aquifer_by_stations[aquifer]['altitude_diff'][-(2 * day_len + horizon):-(horizon)].values})
        X_train = temp[['a', 'b']][:-(day_len + horizon)]
        y_train = temp['target'][horizon:-day_len]

        X_test = temp[['a', 'b']][-(day_len + horizon):-horizon]
        y_test = temp['target'][-day_len:]
        
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()

        # Scale the features
        X_train = pd.DataFrame(scaler_X.fit_transform(X_train), columns=X_train.columns)
        X_test = pd.DataFrame(scaler_X.transform(X_test), columns=X_test.columns)
        
        y_train = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()

        # Train the model
        model.fit(X_train, y_train)

        # Make predictions
        forecast = model.predict(X_test).tolist()
        # Flatten
        forecast = np.ravel(forecast)

        # Unscale the predictions
        forecast = scaler_y.inverse_transform(forecast.reshape(-1, 1)).ravel()

        # Store to the predictions
        predictions.append(forecast)
        
        # Calculate and save the r2 score
        r2_scores[horizon-1].append(r2_score(y_test, forecast))

    # Store the predictions to the dictionary
    predictions_by_stations[aquifer] = predictions

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(aquifer_by_stations[aquifer]['date'][-200:], aquifer_by_stations[aquifer]['altitude_diff'][-200:], color="royalblue", label="true data")
plt.plot(aquifer_by_stations[aquifer]['date'][-200:], predictions[4][-200:], color="tomato", label="forecast")
#plt.plot(aquifer_by_stations[aquifer]['date'][-200:], predictions[2][-200:], color="green", label="forecast")
#plt.plot(aquifer_by_stations[aquifer]['date'][-200:], predictions[4][-200:], color="grey", label="forecast")
#plt.plot(aquifer_by_stations[aquifer]['date'][-200:], aquifer_by_stations[aquifer]['precipitation_probability_max'][-200:].apply(lambda x: x/20), color="brown", label="forecast")
#plt.plot(aquifer_by_stations[aquifer]['date'][-200:], aquifer_by_stations[aquifer]['precipitation'][-200:].apply(lambda x: x/130), color="olive", label="forecast")
plt.plot(aquifer_by_stations[aquifer]['date'][-200:], predictions_by_stations[85064][0][-200:], color="grey", label="forecast")
#plt.savefig('../data/interim/plot.svg', format='svg')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Calculate the average r2 score
r2_average =  []
std_dev = []

for i in range(5):
    r2_average.append(np.mean(r2_scores[i]))
    std_dev.append(np.std(r2_scores[i]))

In [ ]:
r2_average